# 🎯 06 — Retention Prioritization
## Priority Score = Value × Churn Risk × Net Benefit

The centerpiece of the analysis: combines customer value, churn risk, and intervention economics into a transparent Retention Priority Score.

---

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
import config

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Run Retention Prioritization

In [1]:
from src import retention_priority

churn_data = pd.read_csv(os.path.join(config.DATA_PROCESSED, 'customer_churn.csv'))
final_data = retention_priority.run(customer_data=churn_data)

ImportError: cannot import name 'retention_priority' from 'src' (unknown location)

## 2. Decision Matrix Heatmap

In [ ]:
# Create heatmap of value tier × risk tier
pivot = final_data.groupby(['value_tier', 'risk_tier']).agg(
    count=('customer_id', 'count'),
    avg_net_benefit=('expected_net_benefit', 'mean'),
).reset_index()

heatmap_data = pivot.pivot(index='value_tier', columns='risk_tier', values='count').fillna(0)
# Reorder
tier_order = ['High Value', 'Medium Value', 'Low Value']
risk_order = ['Low Risk', 'Medium Risk', 'High Risk']
heatmap_data = heatmap_data.reindex(index=tier_order, columns=risk_order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt=',.0f', cmap='YlOrRd', ax=ax,
            linewidths=2, linecolor='white', cbar_kws={'label': 'Customer Count'})
ax.set_title('Decision Matrix: Value Tier × Risk Level', fontsize=14, fontweight='bold')
ax.set_ylabel('Value Tier')
ax.set_xlabel('Risk Tier')
plt.tight_layout()
plt.show()

## 3. Top Retention Targets

In [ ]:
top_20 = final_data.nsmallest(20, 'priority_rank')[
    ['priority_rank', 'customer_id', 'customer_profit', 'churn_prob',
     'expected_net_benefit', 'retention_priority_score', 'rfm_segment',
     'value_tier', 'recommended_action']
].round(2)

display(top_20)

## 4. ROI Analysis

In [ ]:
positive_roi = final_data[final_data['expected_net_benefit'] > 0]
negative_roi = final_data[final_data['expected_net_benefit'] <= 0]

print(f"Customers with POSITIVE ROI: {len(positive_roi):,} ({len(positive_roi)/len(final_data)*100:.1f}%)")
print(f"Customers with NEGATIVE ROI: {len(negative_roi):,} ({len(negative_roi)/len(final_data)*100:.1f}%)")
print(f"\nTotal budget if targeting positive ROI only: {config.CURRENCY_SYMBOL} {positive_roi['intervention_cost'].sum():,.0f}")
print(f"Expected return: {config.CURRENCY_SYMBOL} {positive_roi['expected_net_benefit'].sum():,.0f}")
print(f"Expected ROI: {positive_roi['expected_net_benefit'].sum() / positive_roi['intervention_cost'].sum() * 100:.0f}%")

# Action distribution
fig, ax = plt.subplots(figsize=(12, 5))
action_counts = final_data['recommended_action'].value_counts()
action_counts.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(action_counts)))
ax.set_title('Recommended Actions Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Customer Count')
plt.tight_layout()
plt.show()

---
**✅ Retention prioritization complete!** Proceed to `07_powerbi_export.ipynb`.